In [334]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler


In [335]:
los_cleaned_df=pd.read_csv('Length_cleaned.csv', index_col=False)
los_cleaned_df.drop(columns='respiration', inplace=True)
los_cleaned_df

,eid,rcount,gender,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,...,neutrophils,sodium,glucose,bloodureanitro,creatinine,bmi,pulse,secondarydiagnosisnonicd9,facid,lengthofstay
0,1,0,F,0,0,0,0,0,0,0,...,14200.0,140.361132,192.476918,12.0,1.390722,30.432418,96,4,B,3
1,2,5,F,0,0,0,0,0,0,0,...,4100.0,136.731692,94.078507,8.0,0.943164,28.460516,61,1,A,7
2,3,1,F,0,0,0,0,0,0,0,...,8900.0,133.058514,130.530524,12.0,1.065750,28.843812,64,2,B,3
3,4,0,F,0,0,0,0,0,0,0,...,9400.0,138.994023,163.377028,12.0,0.906862,27.959007,76,1,A,1
4,5,0,F,0,0,0,1,0,1,0,...,9050.0,138.634836,94.886654,11.5,1.242854,30.258927,67,2,E,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99864,99996,3,M,0,0,0,0,0,0,0,...,9300.0,132.614977,171.422555,12.0,0.650323,30.063069,80,1,B,6
99865,99997,0,M,0,0,0,0,0,0,0,...,9300.0,138.327320,122.342450,12.0,1.521424,28.969548,61,1,B,1
99866,99998,1,M,0,0,1,0,0,0,0,...,7700.0,136.695905,108.288106,12.0,1.025677,26.354919,61,1,C,4
99867,99999,0,M,0,0,0,0,0,0,1,...,8200.0,135.980516,111.750731,16.0,1.035400,29.193462,59,1,B,4


In [336]:
X=los_cleaned_df.drop(columns=['lengthofstay'])
y=los_cleaned_df['lengthofstay']


In [337]:
X_train, X_test, y_train, y_test=train_test_split(X, y, random_state=23, test_size=0.33)

#### Encoding categorical columns

##### Gender M/F

In [338]:
los_cleaned_df['gender']

0        F
1        F
2        F
3        F
4        F
        ..
99864    M
99865    M
99866    M
99867    M
99868    F
Name: gender, Length: 99869, dtype: object

In [339]:
enc=OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
enc_transform=enc.fit_transform(los_cleaned_df[['gender']])
enc_transform

,gender_F,gender_M
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0
...,...,...
99864,0.0,1.0
99865,0.0,1.0
99866,0.0,1.0
99867,0.0,1.0


In [340]:
los_cleaned_df=pd.concat([los_cleaned_df, enc_transform], axis=1).drop(columns=['gender'])

In [341]:
los_cleaned_df

,eid,rcount,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,psychother,...,glucose,bloodureanitro,creatinine,bmi,pulse,secondarydiagnosisnonicd9,facid,lengthofstay,gender_F,gender_M
0,1,0,0,0,0,0,0,0,0,0,...,192.476918,12.0,1.390722,30.432418,96,4,B,3,1.0,0.0
1,2,5,0,0,0,0,0,0,0,0,...,94.078507,8.0,0.943164,28.460516,61,1,A,7,1.0,0.0
2,3,1,0,0,0,0,0,0,0,0,...,130.530524,12.0,1.065750,28.843812,64,2,B,3,1.0,0.0
3,4,0,0,0,0,0,0,0,0,0,...,163.377028,12.0,0.906862,27.959007,76,1,A,1,1.0,0.0
4,5,0,0,0,0,1,0,1,0,0,...,94.886654,11.5,1.242854,30.258927,67,2,E,4,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99864,99996,3,0,0,0,0,0,0,0,0,...,171.422555,12.0,0.650323,30.063069,80,1,B,6,0.0,1.0
99865,99997,0,0,0,0,0,0,0,0,0,...,122.342450,12.0,1.521424,28.969548,61,1,B,1,0.0,1.0
99866,99998,1,0,0,1,0,0,0,0,0,...,108.288106,12.0,1.025677,26.354919,61,1,C,4,0.0,1.0
99867,99999,0,0,0,0,0,0,0,1,0,...,111.750731,16.0,1.035400,29.193462,59,1,B,4,0.0,1.0


#### Facility ID
##### Since this is a synthetic dataset, i made sure the facility column wasnt just randomly assigned values without different LOS values. Since the values are different we can find a pattern. 

In [342]:
los_cleaned_df.groupby('facid')['lengthofstay'].mean()

facid
A    3.269475
B    3.282928
C    4.889789
D    4.816450
E    5.154949
Name: lengthofstay, dtype: float64

In [343]:
enc_transform=enc.fit_transform(los_cleaned_df[['facid']])
enc_transform

,facid_A,facid_B,facid_C,facid_D,facid_E
0,0.0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
99864,0.0,1.0,0.0,0.0,0.0
99865,0.0,1.0,0.0,0.0,0.0
99866,0.0,0.0,1.0,0.0,0.0
99867,0.0,1.0,0.0,0.0,0.0


In [344]:
los_cleaned_df=pd.concat([los_cleaned_df, enc_transform], axis=1).drop(columns=['facid'])

In [345]:
los_cleaned_df

,eid,rcount,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,psychother,...,pulse,secondarydiagnosisnonicd9,lengthofstay,gender_F,gender_M,facid_A,facid_B,facid_C,facid_D,facid_E
0,1,0,0,0,0,0,0,0,0,0,...,96,4,3,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,5,0,0,0,0,0,0,0,0,...,61,1,7,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,1,0,0,0,0,0,0,0,0,...,64,2,3,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4,0,0,0,0,0,0,0,0,0,...,76,1,1,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,5,0,0,0,0,1,0,1,0,0,...,67,2,4,1.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99864,99996,3,0,0,0,0,0,0,0,0,...,80,1,6,0.0,1.0,0.0,1.0,0.0,0.0,0.0
99865,99997,0,0,0,0,0,0,0,0,0,...,61,1,1,0.0,1.0,0.0,1.0,0.0,0.0,0.0
99866,99998,1,0,0,1,0,0,0,0,0,...,61,1,4,0.0,1.0,0.0,0.0,1.0,0.0,0.0
99867,99999,0,0,0,0,0,0,0,1,0,...,59,1,4,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [346]:
los_cleaned_df.columns

Index(['eid', 'rcount', 'dialysisrenalendstage', 'asthma', 'irondef', 'pneum',
       'substancedependence', 'psychologicaldisordermajor', 'depress',
       'psychother', 'fibrosisandother', 'malnutrition', 'hemo', 'hematocrit',
       'neutrophils', 'sodium', 'glucose', 'bloodureanitro', 'creatinine',
       'bmi', 'pulse', 'secondarydiagnosisnonicd9', 'lengthofstay', 'gender_F',
       'gender_M', 'facid_A', 'facid_B', 'facid_C', 'facid_D', 'facid_E'],
      dtype='object')

#### Standardizing numerical columns

##### Reading the Scikit learn documentation for different scalers that can be used, standard scaler which is the most commonly used scaler, is effected by outliers. Since i have decided to keep outliers for BUN and neutrophils, i have used the ROBUST SCALER. 

In [347]:
numerics=['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose', 'hematocrit', 'neutrophils']
X_train_numerical=X_train[numerics]
X_test_numerical=X_test[numerics]

X_train.drop(columns=['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose', 'hematocrit', 'neutrophils'], inplace=True)
X_test.drop(columns=['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose', 'hematocrit', 'neutrophils'], inplace=True)


In [348]:
print(X_train_numerical.columns)
print(X_test_numerical.columns)

Index(['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose',
       'hematocrit', 'neutrophils'],
      dtype='object')
Index(['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose',
       'hematocrit', 'neutrophils'],
      dtype='object')


In [349]:
scaler=RobustScaler()
X_train_scaled=(scaler.fit_transform(X_train_numerical))
X_test_scaled=scaler.transform(X_test_numerical)

X_train_scaled_df=pd.DataFrame(X_train_scaled, columns=['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose','hematocrit', 'neutrophils'], index=X_train_numerical.index)
X_test_scaled_df=pd.DataFrame(X_test_scaled, columns=['bloodureanitro', 'creatinine', 'bmi', 'pulse', 'sodium', 'glucose','hematocrit', 'neutrophils'], index=X_test_numerical.index)

In [350]:
X_train=pd.concat([X_train, X_train_scaled_df ], axis=1, ignore_index=False)
X_test=pd.concat([X_test, X_test_scaled_df], axis=1)
